Select samples to make LD reference panel

In [ ]:
Download ancestry assignments, QC, and unrelated sample

In [ ]:
%%bash

gcloud storage cp --billing-project=$GOOGLE_PROJECT gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv .

gcloud storage cp --billing-project=$GOOGLE_PROJECT gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv .

gcloud storage cp --billing-project=$GOOGLE_PROJECT gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv .

In [ ]:
import pandas as pd

In [ ]:
anc = pd.read_csv("ancestry_preds.tsv", sep='\t')
qc_flagged = pd.read_csv("flagged_samples.tsv", sep = '\t')
rel_flagged = pd.read_csv("relatedness_flagged_samples.tsv", sep = '\t')

Remove flagged IDs

In [ ]:
flagged_ids = pd.concat([rel_flagged['sample_id'], qc_flagged['s']]).unique()
anc_qc = anc[~anc['research_id'].isin(flagged_ids)]

In [ ]:
group_counts = anc_qc['ancestry_pred_other'].value_counts()
group_counts

Randomly select up to 30,000 from each group

In [ ]:
# https://www.random.org/integers/?num=1&min=1&max=1000000000&col=1&base=10&format=html&rnd=new
random_seed = 491100375

min_sample_size = 3000
max_sample_size = 20000

# Keep assigned continental groups with at least 3000 participants 
groups_keep = group_counts[group_counts >= min_sample_size].index
continental_groups_keep = groups_keep[groups_keep != 'oth']

# Keep participants from retained groups
anc_groups_keep = anc_qc[anc_qc['ancestry_pred_other'].isin(continental_groups_keep)]

anc_sampled_panel = anc_groups_keep.groupby('ancestry_pred_other', group_keys=False).apply(
    lambda x: x.sample(n=min(max_sample_size, len(x)), random_state=random_seed)
)

Make FID/IID sample lists

In [ ]:
samples = anc_sampled_panel.assign(FID=0).loc[:, ['FID', 'research_id',
'ancestry_pred']]
samples = samples.rename(columns={'research_id': 'IID'})

samples.to_csv("ancestry_ref_panel_v8.id", sep='\t', index=False)

# separate file for each group
samples_grouped = samples.groupby('ancestry_pred')

for cluster, cluster_df in samples_grouped:
    # Define the file path with the cluster name
    cluster_ids_path = f'ancestry_ref_panel_v8-{cluster}.id'
    
    cluster_df.to_csv(cluster_ids_path, sep='\t', index=False)